In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict
    )

In [4]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [5]:
doc_id = q['document']
results = text_search(query = q['question'])

In [6]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d['id'] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [7]:
relevance = []
for d in results:
    relevance.append(int(d['id'] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [8]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(query = q['question'])

    relevance = []
    for d in results:
       relevance.append(int(d['id'] == doc_id))

    return relevance

In [9]:
q = ground_truth[0]
print(q['question'])
compute_relevance_text(q)

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [10]:
q = ground_truth[50]
print(q['question'])
compute_relevance_text(q)

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [11]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [12]:
ground_truth_sample = ground_truth[:15]
relevance_total_text =compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [13]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [14]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [18]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [19]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [20]:
cnt / len(example)
# 0.933

0.9333333333333333

In [21]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [22]:
hit_rate(example)
# 0.933

0.9333333333333333

In [24]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

In [25]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [26]:
total_score / len(example)
# 0.822

0.8222222222222222

In [27]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [28]:
mrr(example)
# 0.822

0.8222222222222222

In [29]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr" : mrr(relevance_total),
    }

In [ ]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5756118143459916}

In [31]:
{"hit_rate": 0.899, "mrr": 0.769}

{'hit_rate': 0.899, 'mrr': 0.769}

In [33]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
    )

In [34]:
for boost in {0.5, 1.0, 3.0 ,5.0, 10.0}:
    result = evaluate(
        ground_truth,
        lambda query, boost = boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.6886075949367089, 'mrr': 0.58915611814346}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.6936708860759494, 'mrr': 0.59873417721519}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6506329113924051, 'mrr': 0.5756118143459916}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6379746835443038, 'mrr': 0.5528270042194093}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6253164556962025, 'mrr': 0.5353164556962027}


In [35]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section" : section_boost,
        "answer" : answer_boost,
    }

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
    )

In [36]:
results = []

for question_boost in [1.0,2.0,5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f"answer_boost={answer_boost},"
                f"section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost = question_boost , answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query, 
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question":question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })
            

Evaluating question_boost=1.0,answer_boost=1.0,section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0,answer_boost=1.0,section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0,answer_boost=1.0,section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0,answer_boost=10.0,section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0,answer_boost=10.0,section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [37]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending= False).head(10)

,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.754430,0.648945
33,5.0,10.0,0.1,0.749367,0.648692
34,5.0,10.0,0.2,0.754430,0.648650
3,1.0,2.0,0.1,0.749367,0.647637
35,5.0,10.0,0.5,0.749367,0.647637
19,2.0,4.0,0.2,0.749367,0.647637
4,1.0,2.0,0.2,0.746835,0.645696
20,2.0,4.0,0.5,0.746835,0.644852
7,1.0,4.0,0.2,0.744304,0.634388
6,1.0,4.0,0.1,0.744304,0.634304


In [38]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
    )


